# BookToText: Vietnamese History PDF OCR Pipeline

Self-contained Colab notebook. Downloads Vietnamese history PDFs, runs OCR with **PaddleOCR + VietOCR**, saves per-page YAML output, and pushes to GitHub.

**Runtime:** GPU (T4 or better)

## 1. GPU Check

In [ ]:

import subprocess, sys
print('=== CUDA Version ===')
!nvcc --version 2>/dev/null || echo 'nvcc not found'
print('\n=== GPU Info ===')
!nvidia-smi 2>/dev/null || echo 'nvidia-smi not found'
print(f'\n=== Python: {sys.version} ===')

=== CUDA Version ===
nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0

=== GPU Info ===
Fri Jun 12 19:47:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |         

## 2. Install System Dependencies

In [ ]:

!apt-get update -qq && apt-get install -y -qq poppler-utils > /dev/null 2>&1
!echo "poppler-utils installed"
!pdftoppm -v 2>&1 | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
poppler-utils installed
pdftoppm version 22.02.0


## 3. Install PaddlePaddle GPU

In [ ]:
!pip install paddlepaddle-gpu==3.3.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/

Looking in indexes: https://www.paddlepaddle.org.cn/packages/stable/cu126/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 GB 1.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 38.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 41.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 24.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6/216.6 MB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Fix NCCL conflict: paddlepaddle-gpu pins nvidia-nccl-cu12==2.25.1
# but torch 2.11.0+cu128 needs 2.28.9 (for ncclCommShrink symbol)
# Must restart runtime after this cell for fix to take effect!
!pip install --force-reinstall nvidia-nccl-cu12==2.28.9 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
paddlepaddle-gpu 3.3.0 requires nvidia-nccl-cu12==2.25.1; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-nccl-cu12 2.28.9 which is incompatible.
torch 2.11.0+cu128 requires nvidia-cudnn-cu12==9.19.0.56; platform_system == "Linux", but you have nvidia-cudnn-cu12 9.5.1.17 which is incompatible.
torch 2.11.0+cu128 requires nvidia-cusparselt-cu12==0.7.1; platform_system == "Linux", but you have nvidia-cusparselt-cu12 0.6.3 which is incompatible.


## 4. Install Python Packages

In [ ]:
!pip install paddleocr 'paddlex[ocr]' vietocr pdf2image pypdf pyyaml tqdm click opencv-python-headless -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 8.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 8.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.8/146.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 MB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.9/133.9 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install --upgrade pillow==10.2.0


In [ ]:
from paddleocr import TextDetection, LayoutDetection

In [ ]:
from vietocr.tool.predictor import Predictor
from vietocr.tool.config import Cfg
from PIL import Image as PILImage
import numpy as np, cv2, yaml, unicodedata, os, re, logging, time, glob
from io import BytesIO
from pathlib import Path
from tqdm import tqdm
print('All imports OK')

All imports OK


## 5. Download Books

In [ ]:
!mkdir -p /content/input /content/output
!wget -q --show-progress -O /tmp/vietnam_history_pdf.zip https://pub-3d5ab760174a48fda9acb3630968631e.r2.dev/vietnam_history_pdf.zip
!unzip -o -q /tmp/vietnam_history_pdf.zip -d /content/input/

pdfs = [f for f in os.listdir('/content/input') if f.endswith('.pdf')]
print(f'Found {len(pdfs)} PDF files:')
for pdf in sorted(pdfs):
    size_mb = os.path.getsize(f'/content/input/{pdf}') / (1024*1024)
    print(f'  {pdf} ({size_mb:.1f} MB)')

/tmp/vietnam_histor 100%[===================>]   1.68G  42.0MB/s    in 35s     
Found 15 PDF files:
  Lịch sử Việt Nam tập 01 Từ khởi thủy đến thế kỷ X-Cao Duy Mến-2013.pdf (14.7 MB)
  Lịch sử Việt Nam tập 02 Từ thế kỷ X đến thế kỷ XIV-Trần Thị Vinh-2014.pdf (207.7 MB)
  Lịch sử Việt Nam tập 03 Từ thế kỷ XV đến thế kỷ XVI-Tạ Ngọc Liễn-2017.pdf (240.4 MB)
  Lịch sử Việt Nam tập 04 Từ thế kỷ XVII đến thế kỷ XVIII-Trần Thị Vinh-2017.pdf (163.4 MB)
  Lịch sử Việt Nam tập 05 Từ năm 1802 đến năm 1858-Trương Thị Yến-2017.pdf (177.8 MB)
  Lịch sử Việt Nam tập 06 Từ năm 1858 đến năm 1896-Võ Kim Cương-2017.pdf (119.4 MB)
  Lịch sử Việt Nam tập 07 Từ năm 1897 đến năm 1918-Tạ Thị Thúy-2017.pdf (228.1 MB)
  Lịch sử Việt Nam tập 08 Từ năm 1919 đến năm 1930-Tạ Thị Thúy-2017.pdf (14.1 MB)
  Lịch sử Việt Nam tập 09 Từ năm 1930 đến năm 1945-Tạ Th

## 6. Define Pipeline Code

In [ ]:
# === Logger & FileSystem ===
def get_logger(name):
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(name)s: %(message)s'))
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)
    return logger

class FileInput:
    def __init__(self, name: str, data: bytes):
        self.name = name; self.data = data

class FileSystem:
    def __init__(self, base: str): self.base = base
    def put(self, inp):
        p = Path(self.base) / inp.name
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_bytes(inp.data)
    def ls(self, prefix=None):
        base = Path(self.base)
        if not base.exists(): return []
        files = [str(p.relative_to(base)) for p in base.rglob('*') if p.is_file()]
        if prefix: files = [f for f in files if f.startswith(prefix)]
        return files
    def read(self, name):
        return (Path(self.base) / name).read_bytes()

print('Logger & FileSystem OK')

Logger & FileSystem OK


In [ ]:
# === PDF & Converter ===
from pdf2image import convert_from_bytes, pdfinfo_from_bytes

CHUNK_SIZE = 50  # Convert PDF pages in batches to avoid OOM

class PDF:
    def __init__(self, name, data):
        self._name = name; self._data = data
    def get_name(self): return self._name
    def get_data(self): return self._data
    def get_page_count(self):
        return int(pdfinfo_from_bytes(self._data)['Pages'])

class PyPDFConverter:
    def __init__(self, dpi=300):
        self._dpi = dpi; self._logger = get_logger(self.__class__.__name__)
    def convert(self, pdf, start_page=0, end_page=None, dest=None):
        total_pages = pdf.get_page_count()
        last = end_page if end_page is not None else total_pages - 1
        book = unicodedata.normalize('NFC', pdf.get_name())

        # Determine which pages are missing
        missing = None
        if dest is not None:
            img_dir = f'images/{book}'
            existing = set()
            for item in dest.ls():
                n = unicodedata.normalize('NFC', item)
                if n.startswith(img_dir + '/') and n.endswith('.png'):
                    try: existing.add(int(n.split('/')[-1].replace('.png','')))
                    except: pass
            missing = set(range(start_page, last+1)) - existing
            if not missing:
                self._logger.info(f'Skipping {book}: all pages already converted')
                return []
            self._logger.info(f'{book}: {len(missing)} pages to convert')
        else:
            missing = set(range(start_page, last+1))

        # Convert in chunks with progress bar
        images = []
        pages_sorted = sorted(missing)
        chunks = [pages_sorted[i:i+CHUNK_SIZE] for i in range(0, len(pages_sorted), CHUNK_SIZE)]

        with tqdm(total=len(pages_sorted), desc=f'PDF→IMG {book[:30]}', unit='pg') as pbar:
            for chunk in chunks:
                chunk_start = chunk[0]
                chunk_end = chunk[-1]
                pil_images = convert_from_bytes(
                    pdf.get_data(), dpi=self._dpi,
                    first_page=chunk_start+1,  # pdf2image is 1-indexed
                    last_page=chunk_end+1
                )
                for i, pil in enumerate(pil_images):
                    actual = chunk_start + i
                    if actual not in missing: continue
                    buf = BytesIO(); pil.save(buf, format='PNG'); buf.seek(0)
                    images.append(Image(pdf.get_name(), f'page_{actual}', buf.getvalue()))
                    pbar.update(1)
                # Free memory between chunks
                del pil_images
                import gc; gc.collect()

        self._logger.info(f'{book}: converted {len(images)} pages')
        return images

print('PDF & Converter OK (chunked, with progress)')

PDF & Converter OK (chunked, with progress)


In [ ]:
# === Data Classes ===
class Image:
    def __init__(self, book_name, page_name, data):
        self.book_name = book_name; self.page_name = page_name; self.data = data
    def get_book_name(self): return self.book_name
    def get_page_name(self): return self.page_name
    def get_data(self):
        arr = np.frombuffer(self.data, dtype=np.uint8)
        img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
        if img is None: raise ValueError(f'Failed to decode {self.page_name}')
        return img

class LayoutElement:
    def __init__(self, label, score, bbox):
        self.label = label; self.score = score; self.bbox = bbox

class TextElement:
    def __init__(self, text, confidence, box, label='text'):
        self.text = text; self.confidence = confidence; self.box = box; self.label = label

class DetectedText:
    def __init__(self, box, score):
        self.box = box; self.score = score

class Table:
    def __init__(self, bbox, html, score):
        self.bbox = bbox; self.html = html; self.score = score
    def to_dict(self):
        return {'bbox': self.bbox, 'html': self.html, 'score': self.score}

class Page:
    def __init__(self, doc_name, number, header, content, footer, tables=None):
        self.doc_name = doc_name; self.number = number; self.header = header
        self.content = content; self.footer = footer; self.tables = tables or []
    def to_dict(self):
        return {'doc_name': self.doc_name, 'number': self.number, 'header': self.header,
                'content': self.content, 'footer': self.footer,
                'tables': [t.to_dict() for t in self.tables]}

print('Data Classes OK')

Data Classes OK


In [ ]:
# === Box Merging & Geometry Utils ===
def _merge_boxes_to_lines(boxes, y_overlap_threshold=0.5, x_gap_threshold=50):
    if not boxes: return []
    sorted_boxes = sorted(boxes, key=lambda b: (min(p[1] for p in b), min(p[0] for p in b)))
    lines = []
    for box in sorted_boxes:
        x1 = min(p[0] for p in box); y1 = min(p[1] for p in box)
        x2 = max(p[0] for p in box); y2 = max(p[1] for p in box)
        placed = False
        for line in lines:
            lb = line[-1]
            ly1 = min(p[1] for p in lb); ly2 = max(p[1] for p in lb)
            lh = ly2 - ly1; bh = y2 - y1; mh = min(bh, lh)
            if mh <= 0: continue
            y_overlap = max(0, min(y2, ly2) - max(y1, ly1))
            if y_overlap / mh >= y_overlap_threshold:
                lx2 = max(p[0] for p in lb)
                if x1 - lx2 <= x_gap_threshold:
                    line.append(box); placed = True; break
        if not placed: lines.append([box])
    result = []
    for line in lines:
        line.sort(key=lambda b: min(p[0] for p in b))
        min_x = min(min(p[0] for p in b) for b in line)
        min_y = min(min(p[1] for p in b) for b in line)
        max_x = max(max(p[0] for p in b) for b in line)
        max_y = max(max(p[1] for p in b) for b in line)
        result.append([[min_x, min_y], [max_x, max_y]])
    return result

def _bbox_overlap_ratio(text_quad, layout_bbox):
    xs = [p[0] for p in text_quad]; ys = [p[1] for p in text_quad]
    tx1, tx2 = min(xs), max(xs); ty1, ty2 = min(ys), max(ys)
    lx1, ly1, lx2, ly2 = layout_bbox
    ix1, iy1 = max(tx1, lx1), max(ty1, ly1)
    ix2, iy2 = min(tx2, lx2), min(ty2, ly2)
    if ix1 >= ix2 or iy1 >= iy2: return 0.0
    inter = (ix2 - ix1) * (iy2 - iy1)
    area = (tx2 - tx1) * (ty2 - ty1)
    return inter / area if area else 0.0

# === Page Number Parsing ===
_ROMAN_PATTERN = re.compile(r'^[ivxlcdmIVXLCDM]+$')
def _is_valid_roman(s):
    return bool(re.match(r'^M*(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$', s, re.IGNORECASE))
def _roman_to_int(s):
    if not _is_valid_roman(s): return 0
    vals = {'i':1,'v':5,'x':10,'l':50,'c':100,'d':500,'m':1000}
    total, prev = 0, 0
    for c in reversed(s.lower()):
        v = vals.get(c,0); total += -v if v < prev else v; prev = v
    return total
def _parse_page_number(s):
    if not s: return 0
    s = s.strip()
    if not s: return 0
    if s.isdigit(): return int(s)
    m = re.search(r'\d+', s)
    if m: return int(m.group())
    if _ROMAN_PATTERN.match(s) and _is_valid_roman(s): return _roman_to_int(s)
    return 0

print('Geometry & Parsing Utils OK')

Geometry & Parsing Utils OK


In [ ]:
# === OCR Engine: PaddleVietOCR ===
_MODEL_CACHE = {}

def _detect_device():
    """Returns (paddle_device, torch_device). PaddleX='gpu:0', torch='cuda:0'."""
    env = os.environ.get('BOOKTOTEXT_DEVICE','').strip().lower()
    if env == 'cpu': return 'cpu', 'cpu'
    if env in ('cuda', 'gpu'): return 'gpu:0', 'cuda:0'
    if env == 'mps': return 'cpu', 'mps'
    try:
        import torch
        if torch.cuda.is_available(): return 'gpu:0', 'cuda:0'
        if hasattr(torch.backends,'mps') and torch.backends.mps.is_available(): return 'cpu', 'mps'
    except: pass
    return 'cpu', 'cpu'

class PaddleVietOCR:
    LABEL_MAP = {'text':'text','title':'page_title','table':'table',
                 'figure':'figure','header':'header','footer':'footer',
                 'page_number':'number','footnote':'footnote'}

    def __init__(self, debug=False, padding=4):
        self._debug = debug; self._padding = padding
        self._log = get_logger('OCR')
        paddle_device, torch_device = _detect_device()
        self._paddle_device = paddle_device
        self._torch_device = torch_device

        # Timing stats
        self._stats = {'det': [], 'layout': [], 'rec': [], 'total': [], 'pages': 0}

        # Text detection (server model for higher accuracy)
        self._log.info(f'Loading TextDetection model PP-OCRv5_server_det (device={paddle_device})...')
        t0 = time.time()
        cache_key = f'textdet_{paddle_device}'
        if cache_key not in _MODEL_CACHE:
            _MODEL_CACHE[cache_key] = TextDetection(
                model_name='PP-OCRv5_server_det', device=paddle_device)
        self._detector = _MODEL_CACHE[cache_key]
        self._log.info(f'TextDetection ready ({time.time()-t0:.1f}s)')

        # Layout detection
        self._log.info(f'Loading LayoutDetection model (device={paddle_device})...')
        t0 = time.time()
        lkey = f'layout_{paddle_device}'
        if lkey not in _MODEL_CACHE:
            _MODEL_CACHE[lkey] = LayoutDetection(device=paddle_device)
        self._layout = _MODEL_CACHE[lkey]
        self._log.info(f'LayoutDetection ready ({time.time()-t0:.1f}s)')

        # VietOCR recognition
        self._log.info(f'Loading VietOCR model (device={torch_device})...')
        t0 = time.time()
        config = Cfg.load_config_from_name('vgg_transformer')
        config['cnn']['pretrained'] = True
        config['predictor']['beamsearch'] = True
        config['device'] = torch_device
        vkey = f'vietocr_{torch_device}'
        if vkey not in _MODEL_CACHE:
            _MODEL_CACHE[vkey] = Predictor(config)
        self._recognizer = _MODEL_CACHE[vkey]
        self._log.info(f'VietOCR ready ({time.time()-t0:.1f}s)')

        # Table recognition (lazy init)
        self._table_rec = None
        print(f'OCR Engine ready: Paddle={paddle_device}, VietOCR={torch_device}')

    def print_stats(self):
        n = self._stats['pages']
        if n == 0:
            print('No pages processed yet')
            return
        avg_det = sum(self._stats['det']) / n
        avg_lay = sum(self._stats['layout']) / n
        avg_rec = sum(self._stats['rec']) / n
        avg_total = sum(self._stats['total']) / n
        print(f'\n--- OCR Timing Stats ({n} pages) ---')
        print(f'   PaddleOCR TextDetection:   avg {avg_det*1000:.0f}ms/page')
        print(f'   PaddleOCR LayoutDetection: avg {avg_lay*1000:.0f}ms/page')
        print(f'   VietOCR Recognition:       avg {avg_rec*1000:.0f}ms/page')
        print(f'   Total per page:            avg {avg_total:.2f}s/page')
        print(f'   Throughput:                {n / sum(self._stats["total"]):.1f} pages/s')

    def ocr(self, image):
        img = image.get_data()
        t0 = time.time()

        boxes = self._detect_text_boxes(image)
        t1 = time.time()

        layouts = self._detect_layout(image)
        t2 = time.time()

        texts = self._recognize_texts(img, boxes)
        t3 = time.time()

        self._assign_labels(texts, layouts)
        tables = self._process_tables(img, layouts)
        t4 = time.time()

        # Record stats
        self._stats['det'].append(t1 - t0)
        self._stats['layout'].append(t2 - t1)
        self._stats['rec'].append(t3 - t2)
        self._stats['total'].append(t4 - t0)
        self._stats['pages'] += 1

        # Log every 10 pages
        n = self._stats['pages']
        if n % 10 == 0:
            avg_det = sum(self._stats['det'][-10:]) / 10
            avg_lay = sum(self._stats['layout'][-10:]) / 10
            avg_rec = sum(self._stats['rec'][-10:]) / 10
            print(f'  [pg {n}] det={avg_det*1000:.0f}ms lay={avg_lay*1000:.0f}ms rec={avg_rec*1000:.0f}ms ({len(boxes)} boxes) total={t4-t0:.2f}s')

        return self._build_page(image.get_book_name(), texts, tables)

    def _detect_text_boxes(self, image):
        out = self._detector.predict(image.get_data(), batch_size=1)
        if not out: return []
        res = out[0]
        if self._debug:
            os.makedirs(f'./debug/{image.get_book_name()}/text', exist_ok=True)
            res.save_to_img(f'./debug/{image.get_book_name()}/text/{image.get_page_name()}.png')
        data = res.json['res']
        quads = data.get('dt_polys', [])
        scores = data.get('dt_scores', [])
        rects = []
        for quad in quads:
            x1 = int(min(p[0] for p in quad)); y1 = int(min(p[1] for p in quad))
            x2 = int(max(p[0] for p in quad)); y2 = int(max(p[1] for p in quad))
            rects.append([[x1,y1],[x2,y2]])
        merged = _merge_boxes_to_lines(rects)
        detected = []
        for i, box in enumerate(merged):
            score = max(scores) if scores else 1.0
            detected.append(DetectedText(box, float(score)))
        return detected

    def _detect_layout(self, image):
        out = self._layout.predict(image.get_data())
        if not out: return []
        res = out[0]
        if self._debug:
            os.makedirs(f'./debug/{image.get_book_name()}/layout', exist_ok=True)
            res.save_to_img(f'./debug/{image.get_book_name()}/layout/{image.get_page_name()}.png')
        regions = []
        for box in res.get('boxes', []) or []:
            label = self.LABEL_MAP.get(str(box.get('label','text')).lower(), 'text')
            regions.append(LayoutElement(label, float(box.get('score',0.0)),
                                         list(box.get('coordinate',[0,0,0,0]))))
        return regions

    def _recognize_texts(self, img, detected):
        elements = []
        for det in detected:
            xs = [p[0] for p in det.box]; ys = [p[1] for p in det.box]
            x1 = max(0, int(min(xs)) - self._padding)
            y1 = max(0, int(min(ys)) - self._padding)
            x2 = min(img.shape[1], int(max(xs)) + self._padding)
            y2 = min(img.shape[0], int(max(ys)) + self._padding)
            cropped = img[y1:y2, x1:x2]
            if cropped.size == 0: continue
            try:
                pil = PILImage.fromarray(cv2.cvtColor(cropped, cv2.COLOR_BGR2RGB))
                text = self._recognizer.predict(pil)
                elements.append(TextElement(text, det.score, det.box, 'text'))
            except Exception as e:
                self._log.warning(f'Recognition failed: {e}')
        return elements

    def _assign_labels(self, texts, layouts):
        if not layouts: return
        sorted_layouts = sorted(layouts, key=lambda l: -l.score)
        for elem in texts:
            best_label, best_overlap = 'text', 0.0
            for layout in sorted_layouts:
                overlap = _bbox_overlap_ratio(elem.box, layout.bbox)
                if overlap > best_overlap:
                    best_overlap = overlap; best_label = layout.label
            if best_overlap >= 0.5: elem.label = best_label

    def _process_tables(self, img, layouts):
        tables = []
        for layout in layouts:
            if layout.label != 'table': continue
            if self._table_rec is None:
                self._log.info('Loading TableRecognitionPipelineV2 (first table)...')
                from paddleocr import TableRecognitionPipelineV2
                self._table_rec = TableRecognitionPipelineV2()
                self._log.info('TableRecognition ready')
            x1,y1,x2,y2 = map(int, layout.bbox)
            timg = img[y1:y2, x1:x2]
            if timg.size == 0: continue
            try:
                out = self._table_rec.predict(timg)
                if not out:
                    tables.append(Table(list(layout.bbox), '', layout.score))
                    continue
                html = '\n'.join(t.get('pred_html','') for t in out[0].get('table_res_list',[]) if t.get('pred_html'))
                tables.append(Table(list(layout.bbox), html, layout.score))
            except Exception as e:
                self._log.warning(f'Table OCR failed: {e}')
                tables.append(Table(list(layout.bbox), '', layout.score))
        return tables

    def _build_page(self, doc_name, elements, tables):
        header, footer, content = [], [], []
        page_number = ''
        elems = sorted(elements, key=lambda e: min(p[1] for p in e.box))
        for elem in elems:
            if elem.label == 'header': header.append(elem.text)
            elif elem.label == 'footer': footer.append(elem.text)
            elif elem.label == 'number': page_number = elem.text
            elif elem.label == 'page_title': content.insert(0, elem.text)
            elif elem.label == 'table': pass
            else: content.append(elem.text)
        return Page(doc_name, _parse_page_number(page_number),
                  ' '.join(header), content, ' '.join(footer), tables)

print('PaddleVietOCR OK (with timing stats)')

PaddleVietOCR OK (with timing stats)


In [ ]:
# === Output & Pipeline Orchestration (Parallel: Convert + OCR overlap) ===
import threading
import queue

class YAMLOutput:
    def __init__(self, fs): self.fs = fs; self._lock = threading.Lock()
    def add(self, page):
        data = yaml.dump(page.to_dict(), allow_unicode=True, sort_keys=False)
        with self._lock:
            self.fs.put(FileInput(f'yaml/{page.doc_name}/{page.number}.yaml', data.encode('utf-8')))
    def scan_existing_pages(self, dest):
        result = {}
        for item in dest.ls():
            n = unicodedata.normalize('NFC', item)
            if n.startswith('yaml/') and n.endswith('.yaml'):
                parts = n.split('/')
                if len(parts) >= 3:
                    try:
                        result.setdefault(parts[1], set()).add(int(parts[-1].replace('.yaml','')))
                    except: pass
        return result

def _need_continue(files, dest, output):
    existing = output.scan_existing_pages(dest)
    done = set(existing.keys())
    need = []
    for f in files:
        book = unicodedata.normalize('NFC', f)
        if book not in done:
            need.append({'name': f, 'status': 'begin', 'ocr_checkpoint': -1})
        else:
            pages = existing.get(book, set())
            last = max(pages) if pages else -1
            need.append({'name': f, 'status': 'incomplete', 'ocr_checkpoint': last})
    return need

def _image_to_png_bytes(img_data):
    if isinstance(img_data, np.ndarray):
        if len(img_data.shape) == 3:
            mode = 'RGB' if img_data.shape[2] == 3 else 'RGBA'
            pil = PILImage.fromarray(img_data, mode=mode)
        else:
            pil = PILImage.fromarray(img_data, mode='L')
    else: pil = img_data
    buf = BytesIO(); pil.save(buf, format='PNG'); buf.seek(0)
    return buf.getvalue()

def _load_existing_images(dest, book_name, start_page, end_page):
    log = get_logger('Pipeline')
    images = []
    book = unicodedata.normalize('NFC', book_name)
    prefix = f'images/{book}/'
    files = []
    for item in dest.ls():
        n = unicodedata.normalize('NFC', item)
        if n.startswith(prefix) and n.endswith('.png'):
            try:
                num = int(n.split('/')[-1].replace('.png',''))
                files.append((num, n))
            except: pass
    files.sort(key=lambda x: x[0])
    for num, path in files:
        if start_page is not None and num < start_page: continue
        if end_page is not None and num > end_page: continue
        try:
            img_data = dest.read(path)
            images.append(Image(book_name, f'page_{num}', img_data))
        except Exception as e:
            log.warning(f'Failed to load {path}: {e}')
    log.info(f'Loaded {len(images)} existing images for {book[:40]}')
    return images

# ---------- OCR Worker (runs in separate thread, owns GPU) ----------
def _ocr_worker(ocr_queue, dest, ocr, output, stats):
    """Consumer: picks up (file_info, images) from queue, runs OCR on GPU."""
    log = get_logger('OCR-Worker')
    while True:
        item = ocr_queue.get()
        if item is None:  # Poison pill
            ocr_queue.task_done()
            break

        file_info, images, start_idx = item
        book = file_info['name'][:50]
        total = len(images)

        if total == 0:
            log.info(f'[OCR] No pages to OCR for {book}')
            ocr_queue.task_done()
            continue

        log.info(f'[OCR] Starting {book} ({total} pages from idx {start_idx})')
        t_book = time.time()

        with tqdm(range(total), desc=f'OCR {book[:35]}', unit='pg') as pbar:
            for j in pbar:
                i = start_idx + j
                t_page = time.time()
                image = images[j]

                # Save image to disk
                img_data = _image_to_png_bytes(image.get_data())
                dest.put(FileInput(f'images/{file_info["name"]}/{i}.png', img_data))

                # OCR on GPU
                page = ocr.ocr(image)
                if page:
                    if page.number == 0: page.number = i
                    output.add(page)

                elapsed = time.time() - t_page
                pbar.set_postfix(t=f'{elapsed:.1f}s', pg=i)

        elapsed_book = time.time() - t_book
        stats['books_done'] = stats.get('books_done', 0) + 1
        stats['pages_done'] = stats.get('pages_done', 0) + total
        log.info(f'[OCR] Done {book} in {elapsed_book:.1f}s ({total} pages, {elapsed_book/max(total,1):.2f}s/pg)')
        ocr_queue.task_done()

# ---------- Producer: PDF->Image (CPU-bound) ----------
def _convert_book(file_info, src, dest, converter, start_page, end_page):
    """Reads PDF and converts to images. Returns (images, ocr_start_idx)."""
    log = get_logger('Converter')
    book = file_info['name'][:50]

    log.info(f'[READ] {book}...')
    t0 = time.time()
    file_bytes = src.read(file_info['name'])
    log.info(f'[READ] Done ({len(file_bytes)/1024/1024:.1f}MB, {time.time()-t0:.1f}s)')

    pdf = PDF(file_info['name'], file_bytes)
    effective_start = start_page if start_page is not None else 0
    effective_end = end_page

    log.info(f'[CONVERT] {book}...')
    t0 = time.time()
    images = converter.convert(pdf, effective_start, effective_end, dest)
    log.info(f'[CONVERT] Done: {len(images)} images ({time.time()-t0:.1f}s)')

    if len(images) == 0:
        log.info(f'[LOAD] Loading existing images for {book}...')
        t0 = time.time()
        images = _load_existing_images(dest, file_info['name'], effective_start, effective_end)
        log.info(f'[LOAD] Done: {len(images)} images ({time.time()-t0:.1f}s)')

    # Determine OCR start based on checkpoint
    ocr_start = 0 if file_info['status'] == 'begin' else file_info.get('ocr_checkpoint', -1) + 1
    start = max(ocr_start, effective_start)
    end_val = effective_end if effective_end is not None else len(images)
    end_idx = min(end_val + 1, len(images))

    # Slice images to only what needs OCR
    if start > 0 or end_idx < len(images):
        images = images[start:end_idx]

    return images, start

def run_pipeline(src, dest, converter, ocr, output, start_page=None, end_page=None):
    log = get_logger('Pipeline')
    log.info('Listing input PDFs...')
    pdfs = src.ls()
    log.info(f'Found {len(pdfs)} PDFs')

    to_convert = _need_continue(pdfs, dest, output)
    if not to_convert:
        log.info('Nothing to process')
        return

    print(f'\n{"="*60}')
    print(f'{len(to_convert)} books to process (parallel convert+OCR)')
    print(f'{"="*60}')

    # Queue: holds (file_info, images, start_idx) for OCR worker
    ocr_queue = queue.Queue(maxsize=30)  # Buffer up to 2 books ahead
    stats = {}

    # Start OCR worker thread (owns GPU for inference)
    worker = threading.Thread(target=_ocr_worker, args=(ocr_queue, dest, ocr, output, stats), daemon=True)
    worker.start()

    # Main thread: convert books (CPU) and feed to OCR queue
    t_total = time.time()
    for idx, f in enumerate(to_convert, 1):
        print(f'\n{"_"*60}')
        print(f'[{idx}/{len(to_convert)}] CONVERT: {f["name"][:60]}')
        print(f'  Status: {f["status"]}, checkpoint: {f.get("ocr_checkpoint",-1)}')
        print(f'{"_"*60}')

        t_book = time.time()
        images, ocr_start = _convert_book(f, src, dest, converter, start_page, end_page)
        print(f'  Convert done in {time.time()-t_book:.1f}s, {len(images)} pages -> OCR queue')

        if len(images) > 0:
            # This blocks if queue is full (OCR still working on previous book)
            ocr_queue.put((f, images, ocr_start))
        else:
            print(f'  No pages to OCR, skipping')

    # Send poison pill and wait for OCR to finish
    ocr_queue.put(None)
    worker.join()

    elapsed = time.time() - t_total
    print(f'\n{"="*60}')
    print(f'Pipeline complete: {stats.get("books_done",0)} books, {stats.get("pages_done",0)} pages')
    print(f'Total time: {elapsed:.1f}s')
    print(f'{"="*60}')

print('Output & Pipeline OK (parallel: convert overlaps with OCR)')

Output & Pipeline OK (parallel: convert overlaps with OCR)


## 7. Run OCR Pipeline

In [ ]:
_MODEL_CACHE.clear()
print('Model cache cleared')

Model cache cleared


In [ ]:
src = FileSystem('/content/input')
dest = FileSystem('/content/output')
converter = PyPDFConverter(dpi=300)
ocr_engine = PaddleVietOCR(debug=False)
output = YAMLOutput(dest)

# Show device info
paddle_dev, torch_dev = _detect_device()
print(f'Paddle device: {paddle_dev}')
print(f'VietOCR device: {torch_dev}')
import torch, paddle
print(f'torch.cuda.is_available(): {torch.cuda.is_available()}')
print(f'paddle.is_compiled_with_cuda(): {paddle.is_compiled_with_cuda()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

t0 = time.time()
run_pipeline(src, dest, converter, ocr_engine, output)
print(f'\nTotal time: {time.time()-t0:.1f}s ({(time.time()-t0)/60:.1f}m)')

# Print timing breakdown
ocr_engine.print_stats()

yaml_files = glob.glob('/content/output/yaml/**/*.yaml', recursive=True)
print(f'Generated {len(yaml_files)} YAML files')

2026-06-12 19:54:31,590 INFO OCR: Loading TextDetection model PP-OCRv5_server_det (device=gpu:0)...
INFO:OCR:Loading TextDetection model PP-OCRv5_server_det (device=gpu:0)...
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-OCRv5_server_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv5_server_det`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
2026-06-12 19:54:40,556 INFO OCR: TextDetection ready (9.0s)
INFO:OCR:TextDetection ready (9.0s)
2026-06-12 19:54:40,557 INFO OCR: Loading LayoutDetection model (device=gpu:0)...
INFO:OCR:Loading LayoutDetection model (device=gpu:0)...
Using official model (PP-DocLayout_plus-L), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-DocLayout_plus-L`.


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

2026-06-12 19:54:44,522 INFO OCR: LayoutDetection ready (4.0s)
INFO:OCR:LayoutDetection ready (4.0s)
2026-06-12 19:54:44,523 INFO OCR: Loading VietOCR model (device=cuda:0)...
INFO:OCR:Loading VietOCR model (device=cuda:0)...


...

2026-06-12 20:13:07,623 INFO PyPDFConverter: Lịch sử Việt Nam tập 05 Từ năm 1802 đến năm 1858-Trương Thị Yến-2017.pdf: 722 pages to convert
INFO:PyPDFConverter:Lịch sử Việt Nam tập 05 Từ năm 1802 đến năm 1858-Trương Thị Yến-2017.pdf: 722 pages to convert

PDF→IMG Lịch sử Việt Nam tập 05 Từ năm:  13%|█▎        | 97/722 [01:15<03:32,  2.93pg/s]

  [pg 60] det=140ms lay=105ms rec=13911ms (34 boxes) total=13.12s



OCR Lịch sử Việt Nam tập 04 Từ:  11%|█         | 70/650 [13:27<1:57:03, 12.11s/pg, pg=69, t=13.4s]

  [pg 70] det=138ms lay=102ms rec=11333ms (32 boxes) total=12.86s



OCR Lịch sử Việt Nam tập 04 Từ:  12%|█▏        | 77/650 [14:50<1:34:12,  9.86s/pg, pg=76, t=8.9s]

## 8. View Sample Output

In [ ]:
# import shutil, os
# if os.path.exists('/content/output'):
#     shutil.rmtree('/content/output')
#     os.makedirs('/content/output')
#     print('Output cleaned')
# else:
#     os.makedirs('/content/output')
#     print('Output dir created')

In [ ]:
yaml_files = sorted(glob.glob('/content/output/yaml/**/*.yaml', recursive=True))
if yaml_files:
    with open(yaml_files[0]) as f:
        sample = yaml.safe_load(f)
    print(yaml.dump(sample, allow_unicode=True, sort_keys=False))
else:
    print('No YAML output found yet.')

## 9. Package Results

In [ ]:
!cd /content && tar -czf /content/booktotext_output.tar.gz output/
!ls -lh /content/booktotext_output.tar.gz

## 10. Upload to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

DRIVE_DEST = '/content/drive/MyDrive/booktotext_output'
os.makedirs(DRIVE_DEST, exist_ok=True)

# Copy output folder to Drive
shutil.copytree('/content/output', DRIVE_DEST, dirs_exist_ok=True)

# Also copy the tarball
shutil.copy2('/content/booktotext_output.tar.gz', '/content/drive/MyDrive/booktotext_output.tar.gz')

print(f'Uploaded to Google Drive: {DRIVE_DEST}')
print(f'Tarball: /content/drive/MyDrive/booktotext_output.tar.gz')